<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/03_train_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive[cite: 4]
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn[cite: 4]

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab[cite: 4]

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive[cite: 4]
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)[cite: 3]
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)[cite: 3, 4]
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại[cite: 3]
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device)                 # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)
print('Thư mục dữ liệu processed tồn tại:', os.path.exists(PROCESSED_DATA_DIR)) # Kiểm tra thư mục dữ liệu đã sẵn sàng chưa[cite: 3]

# ===== THÊM ĐOẠN NÀY VÀO CELL 1 ĐỂ CỐ ĐỊNH KẾT QUẢ =====
#import random
#import numpy as np
#import torch

#def set_seed(seed=42):
#    random.seed(seed)
#    np.random.seed(seed)
#    torch.manual_seed(seed)
#    torch.cuda.manual_seed(seed)
#    torch.cuda.manual_seed_all(seed)
#    torch.backends.cudnn.deterministic = True
#    torch.backends.cudnn.benchmark = False

#set_seed(42)  # Cố định hạt giống ngẫu nhiên là 42

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda
Thư mục dữ liệu processed tồn tại: True


In [3]:
# ===== CELL 2: ĐỊNH NGHĨA PYTORCH DATASET =====
from torch.utils.data import Dataset, DataLoader # Import hai lớp quản lý Dataset và nạp dữ liệu theo Batch
import numpy as np                               # Import NumPy để nạp các file lưu dưới dạng mảng .npy[cite: 3]

class PMSMDataset(Dataset):
    """
    Lớp Custom Dataset dùng để nạp Input (X) và Target (Y) cho mô hình MLP thuần dữ liệu.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp mảng Input đã chuẩn hóa (StandardScaler): ud, uq, Id, Iq, nm[cite: 3]
        self.X = np.load(os.path.join(data_dir, f'X_{split}.npy'))

        # Nạp mảng Target đã chuẩn hóa (StandardScaler): Rs, Ld, Lq[cite: 3]
        self.Y = np.load(os.path.join(data_dir, f'Y_{split}.npy'))

        # Chuyển đổi dữ liệu từ dạng NumPy Array sang Tensor của PyTorch (kiểu float32)
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)

    def __len__(self):
        # Trả về tổng số lượng mẫu dữ liệu có trong tập dữ liệu
        return len(self.X)

    def __getitem__(self, idx):
        # Trả về bộ 2 giá trị tương ứng với chỉ số (index) idx: Input và Target
        return self.X[idx], self.Y[idx]

# Khởi tạo 3 tập dữ liệu từ thư mục processed trên Google Drive[cite: 3]
train_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='train')       # Tập Train (Data1, Data2, Data3 - 90,000 mẫu)[cite: 3]
val_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='validation') # Tập Validation (Data4 - 30,000 mẫu)[cite: 3]
test_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='test')         # Tập Test (Data5 - 30,000 mẫu)[cite: 3]

# Khởi tạo DataLoader để chia dữ liệu thành từng Batch (kích thước 128 mẫu/batch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)  # Trộn ngẫu nhiên tập Train khi nạp
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)     # Giữ nguyên thứ tự tập Validation
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)   # Giữ nguyên thứ tự tập Test

print(f'Số mẫu Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}')

Số mẫu Train: 90000 | Validation: 30000 | Test: 30000


In [4]:
# ===== CELL 3: KIẾN TRÚC MẠNG MLP =====
import torch.nn as nn # Import module chứa các lớp mạng Neural của PyTorch

class PMSM_MLP(nn.Module):
    """
    Kiến trúc Mạng Multi-Layer Perceptron (MLP) thuần dữ liệu (Data-driven).
    Nhận 5 nút đầu vào [ud, uq, Id, Iq, nm] và dự đoán 3 nút đầu ra [Rs, Ld, Lq].[cite: 2, 3]
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_MLP, self).__init__()

        # Xây dựng các lớp liên kết đầy đủ (Fully Connected / Dense Layers)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # Lớp đầu vào: 5 nút -> 128 nút ẩn
            nn.Tanh(),                          # Hàm kích hoạt Tanh tạo thuộc tính phi tuyến
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 1: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 2: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, output_dim)   # Lớp đầu ra: 128 -> 3 tham số ước lượng [Rs, Ld, Lq][cite: 2, 3]
        )

    def forward(self, x):
        # Hàm lan truyền tiến (Forward Pass) truyền tensor x qua các lớp mạng
        return self.network(x)

# Khởi tạo mô hình MLP và chuyển toàn bộ thông số lên GPU (cuda)
model = PMSM_MLP(input_dim=5, output_dim=3, hidden_dim=128).to(device)
print(model) # In cấu trúc mạng MLP ra màn hình

PMSM_MLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [5]:
# ===== CELL 4: HUẤN LUYỆN MÔ HÌNH MLP =====
import torch.optim as optim # Import bộ tối ưu hóa trọng số Optimizer

# 1. Định nghĩa Hàm Loss và Optimizer
criterion = nn.MSELoss()                             # Hàm sai số bình phương trung bình (Mean Squared Error)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Thuật toán tối ưu Adam với tốc độ học (Learning Rate) = 1e-3

epochs = 100                    # Tổng số lượt huấn luyện toàn bộ tập dữ liệu (100 Epochs)
best_val_loss = float('inf')    # Khởi tạo kỷ lục Validation Loss ban đầu là vô cùng lớn

print("=== BẮT ĐẦU HUẤN LUYỆN MLP ===")

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model.train() # Bật chế độ huấn luyện (Training Mode)
    train_loss = 0.0

    # Lặp qua từng Batch dữ liệu (128 mẫu) trong train_loader
    for inputs, targets in train_loader:
        # Chuyển dữ liệu của Batch hiện tại lên thiết bị tính toán (GPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Đặt lại gradient của các trọng số về 0 trước khi tính đạo hàm mới
        optimizer.zero_grad()

        # Lan truyền tiến: Mô hình MLP dự đoán đầu ra từ inputs
        outputs = model(inputs)

        # Tính toán sai số MSE giữa dự đoán (outputs) và nhãn thực tế (targets)
        loss = criterion(outputs, targets)

        # Lan truyền ngược (Backpropagation): Tính gradient sai số theo từng trọng số W, b
        loss.backward()

        # Cập nhật trọng số mạng bằng thuật toán Adam
        optimizer.step()

        # Tích lũy tổng sai số trong Epoch hiện tại
        train_loss += loss.item() * inputs.size(0)

    # Tính sai số MSE trung bình trên toàn bộ tập Train (90,000 mẫu)[cite: 3]
    train_loss /= len(train_dataset)

    # --- GIAI ĐOẠN 2: GIÁM SÁT & ĐÁNH GIÁ (VALIDATION) ---
    model.eval() # Bật chế độ đánh giá (Evaluation Mode)
    val_loss = 0.0

    with torch.no_grad(): # Tắt cơ chế tự động tính gradient để tiết kiệm GPU VRAM và tăng tốc
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets) # Tính sai số MSE trên tập Validation[cite: 3]
            val_loss += loss.item() * inputs.size(0)

    # Tính sai số MSE trung bình trên toàn bộ tập Validation (30,000 mẫu)[cite: 3]
    val_loss /= len(val_dataset)

    # In kết quả tiến độ huấn luyện sau mỗi 10 Epochs
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{epochs}] | Train Loss (MSE): {train_loss:.6f} | Val Loss (MSE): {val_loss:.6f}")

    # 4. Tự động lưu bản trọng số (Checkpoint) tốt nhất vào Google Drive[cite: 3, 4]
    if val_loss < best_val_loss:
        best_val_loss = val_loss # Cập nhật lại kỷ lục Validation Loss nhỏ nhất
        checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth')

        # Lưu file .pth chứa trạng thái trọng số tốt nhất lên Drive[cite: 3, 4]
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, checkpoint_path)

print(f"\n--> Đã hoàn thành train! Mô hình tốt nhất đã lưu tại: {CHECKPOINT_DIR}/best_mlp_model.pth")

=== BẮT ĐẦU HUẤN LUYỆN MLP ===
Epoch [  1/100] | Train Loss (MSE): 0.302280 | Val Loss (MSE): 0.362781
Epoch [ 10/100] | Train Loss (MSE): 0.247524 | Val Loss (MSE): 0.294875
Epoch [ 20/100] | Train Loss (MSE): 0.205076 | Val Loss (MSE): 0.337322
Epoch [ 30/100] | Train Loss (MSE): 0.188819 | Val Loss (MSE): 0.325189
Epoch [ 40/100] | Train Loss (MSE): 0.175614 | Val Loss (MSE): 0.344493
Epoch [ 50/100] | Train Loss (MSE): 0.168771 | Val Loss (MSE): 0.327831
Epoch [ 60/100] | Train Loss (MSE): 0.165586 | Val Loss (MSE): 0.384209
Epoch [ 70/100] | Train Loss (MSE): 0.158845 | Val Loss (MSE): 0.315788
Epoch [ 80/100] | Train Loss (MSE): 0.149486 | Val Loss (MSE): 0.322532
Epoch [ 90/100] | Train Loss (MSE): 0.137549 | Val Loss (MSE): 0.365279
Epoch [100/100] | Train Loss (MSE): 0.129278 | Val Loss (MSE): 0.396716

--> Đã hoàn thành train! Mô hình tốt nhất đã lưu tại: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/checkpoints/best_mlp_model.pth


In [6]:
# ===== CELL 5: ĐÁNH GIÁ TRÊN TẬP TEST =====
import joblib # Import joblib để nạp file lưu bộ giải chuẩn hóa Scaler[cite: 3]

# 1. Nạp lại bộ Scaler của Target đã được fit ở bước tiền xử lý[cite: 3]
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# 2. Nạp bản trọng số mô hình tốt nhất đã lưu trên Google Drive[cite: 3, 4]
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth'))
model.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trọng số tối ưu
model.eval()                                           # Chuyển mô hình sang chế độ dự đoán

# 3. Dự đoán toàn bộ tập Test (Data5 - 30,000 mẫu)[cite: 3]
all_preds = []   # Mảng lưu kết quả dự đoán đã chuẩn hóa
all_targets = [] # Mảng lưu nhãn thực tế đã chuẩn hóa

with torch.no_grad(): # Tắt cơ chế tính gradient
    for inputs, targets in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu Batch lên GPU
        outputs = model(inputs)                       # Mạng MLP dự đoán
        all_preds.append(outputs.cpu().numpy())       # Chuyển kết quả dự đoán từ GPU về CPU dưới dạng NumPy Array
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy Array

# Ghép danh sách các Batch thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# 4. Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực tế (Ohm cho Rs, Henry cho Ld và Lq)[cite: 3]
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# 5. Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
# Công thức: MAPE = Mean( | (Thực_tế - Dự_đoán) / Thực_tế | ) * 100%
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình: {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình: {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình: {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===
Sai số Rs trung bình: 7.00%
Sai số Ld trung bình: 0.03%
Sai số Lq trung bình: 0.07%
